In [1]:
from kaggle_secrets import UserSecretsClient
secret_label = "Token"
token = UserSecretsClient().get_secret(secret_label)

! git clone https://{token}@github.com/PietroGhersetich/RecSys_challenge_2025

Cloning into 'RecSys_challenge_2025'...


remote: Enumerating objects: 1345, done.


remote: Counting objects: 100% (151/151), done.


remote: Compressing objects: 100% (145/145), done.


remote: Total 1345 (delta 102), reused 12 (delta 6), pack-reused 1194 (from 4)
Receiving objects: 100% (1345/1345), 531.42 MiB | 56.38 MiB/s, done.


Resolving deltas: 100% (774/774), done.


In [2]:
! pip install PyGithub requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/432.7 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━ 399.4/432.7 kB 12.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.4 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.6 MB/s eta 0:00:00


In [3]:
#! cd /kaggle/working/RecSys_challenge_2025 && python run_compile_all_cython.py

In [4]:
! cd /kaggle/working/RecSys_challenge_2025

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import random
import tqdm
import gc
import time
import os
import shutil
import optuna
from optuna.distributions import FloatDistribution, IntDistribution, CategoricalDistribution
import scipy.sparse as sps

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/sample_submission.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv


In [6]:
from github import Github
from github import Auth

auth = Auth.Token(token)
g = Github(auth=auth)

repo = None
for r in g.get_user().get_repos():
    if r.name == 'RecSys_challenge_2025':
        repo = r
        print('Repository found')

Repository found


In [7]:
def upload_file(filepath_kaggle, filepath_github, commit_message):
    try:
        contents = repo.get_contents(filepath_github)
        with open(filepath_kaggle, "rb") as file:
            repo.update_file(contents.path, commit_message, file.read(), contents.sha)
    except Exception:
        with open(filepath_kaggle, "rb") as file:
            repo.create_file(filepath_github, commit_message, file.read())

In [8]:
db_name = 'multvae_study.db'
json_name = 'multvae_best_params.json'

config = {
    'tune_parameters': True,
    'database_name': db_name,
    'database_path': f'/kaggle/working/{db_name}',
    'save_json': True,
    'json_name': json_name,
    'copy_prev_best_params': False,
    'save_github': True,
    'candidates': '20',
    'n_trials': 10
}

In [9]:
try:
    shutil.copyfile(
        f'/kaggle/working/RecSys_challenge_2025/tuning_results/recall@{config["candidates"]}/{config["database_name"]}', 
        config['database_path']
    )
except FileNotFoundError:
    pass # if not present optuna will create it

# Data Import

In [10]:
train_data = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv')
user_ids_test = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv')

In [11]:
data = np.ones(len(train_data))

URM_all = sps.csr_matrix((data, (train_data['row'].values, train_data['col'].values)),
                         shape=(train_data['row'].max() + 1, train_data['col'].max() + 1))

URM_all

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3043058 stored elements and shape (27095, 6969)>

In [12]:
%cd /kaggle/working/RecSys_challenge_2025

/kaggle/working/RecSys_challenge_2025


In [13]:
from Recommenders.BaseRecommender import BaseRecommender
from Recommenders.Neural.MultVAERecommender import MultVAERecommender
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

from Evaluation.Evaluator import EvaluatorHoldout

np.int = int
np.random.seed(42)
np.bool = bool
np.float = float

2025-12-28 09:55:43.979436: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766915744.236356      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766915744.302456      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [14]:
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_all, train_percentage = 0.8)

In [15]:
import sqlite3

def print_all_rows(table_name, database_path):
    connection = sqlite3.connect(database_path)
    cursor = connection.cursor()

    try:
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()

        print(f"All rows from table '{table_name}':")
        for row in rows:
            print(row)

    except sqlite3.Error as e:
        print(f"Error reading data from table '{table_name}': {e}")

    finally:
        cursor.close()
        connection.close()

print_all_rows('studies', config['database_path'])

All rows from table 'studies':
(1, 'multvae_Optimization')
(2, 'multvae_Optimization_v2')
(3, 'multvae_Optimization_v3')
(4, 'multvae_Optimization_noTorch')


In [16]:
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])

def objective_function_multvae(trial):
    
    early_stopping_params = {
        "validation_every_n": 1,
        "stop_on_validation": True,
        "evaluator_object": evaluator,
        "lower_validations_allowed": 20,
        "validation_metric": "RECALL",
        "epochs": 500
    }
    
    # Define hyperparameter search space
    n_layers = trial.suggest_categorical('n_layers', [1, 2])
    layer_1 = trial.suggest_int('layer_1', 20, 500, log=True)
    p_dims = [layer_1]
    if n_layers == 2:
        p_dims.append(trial.suggest_int('layer_2', layer_1, 2000, log=True))
    
    recommender_instance = MultVAERecommender(URM_train)
    recommender_instance.fit(
        learning_rate=trial.suggest_float('learning_rate', 1e-6, 1e-1, log=True),
        batch_size=trial.suggest_categorical('batch_size', [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]),
        dropout=trial.suggest_float('dropout',0.1, 0.7),
        l2_reg=trial.suggest_float('l2_reg', 1e-7, 0.1, log=True),
        anneal_cap=trial.suggest_float('anneal_cap', 1e-2, 1, log=True),
        total_anneal_steps=trial.suggest_int('total_anneal_steps', 1e3, 1e6, log=True),
        p_dims=p_dims,
        **early_stopping_params
    )
    
    # Store training epochs for analysis
    epochs = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    trial.set_user_attr("epochs", epochs)
    
    # Evaluate
    result_df, _ = evaluator.evaluateRecommender(recommender_instance)
    
    return result_df.loc[int(config['candidates'])]["RECALL"]

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions


In [17]:
%%time

if config['tune_parameters']:
    study = optuna.create_study(direction='maximize', study_name='multvae_Optimization_noTorch_v2', 
                                storage=f'sqlite:///{config["database_path"]}', load_if_exists=True)
    print(f"Previous trials found: {len(study.trials)}")
    study.optimize(objective_function_multvae, n_trials=config['n_trials'])

[I 2025-12-28 09:56:05,743] A new study created in RDB with name: multvae_Optimization_noTorch_v2


Previous trials found: 0
MultVAERecommender: Using default Temp folder './result_experiments/__Temp_MultVAERecommender_20/'


I0000 00:00:1766915767.137988      20 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1766915767.138798      20 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


I0000 00:00:1766915767.452529      20 mlir_graph_optimization_pass.cc:401] MLIR V1 optimization pass is not enabled


MultVAERecommender: Saving model in file './result_experiments/__Temp_MultVAERecommender_20/_best_model'
MultVAERecommender: Saving complete


...

MultVAERecommender: Validation begins...


EvaluatorHoldout: Processed 27067 (100.0%) in 15.81 sec. Users per second: 1712


MultVAERecommender: CUTOFF: 20 - PRECISION: 0.1446559, PRECISION_RECALL_MIN_DEN: 0.2180652, RECALL: 0.1794662, MAP: 0.0597200, MAP_MIN_DEN: 0.0856122, MRR: 0.4153220, NDCG: 0.2066382, F1: 0.1601917, HIT_RATE: 0.8808143, ARHR_ALL_HITS: 0.6723717, NOVELTY: 0.0264322, AVERAGE_POPULARITY: 0.4425438, DIVERSITY_MEAN_INTER_LIST: 0.8330225, DIVERSITY_HERFINDAHL: 0.9916496, COVERAGE_ITEM: 0.1368920, COVERAGE_ITEM_HIT: 0.1163725, ITEMS_IN_GT: 0.9991390, COVERAGE_USER: 0.9989666, COVERAGE_USER_HIT: 0.8799040, USERS_IN_GT: 0.9989666, DIVERSITY_GINI: 0.0227623, SHANNON_ENTROPY: 7.6705370, RATIO_DIVERSITY_HERFINDAHL: 0.9924171, RATIO_DIVERSITY_GINI: 0.0841640, RATIO_SHANNON_ENTROPY: 0.6841515, RATIO_AVERAGE_POPULARITY: 2.0822144, RATIO_NOVELTY: 0.1828604, 

MultVAERecommender: New best model found! Updating.
MultVAERecommender: Saving model in file './result_experiments/__Temp_MultVAERecommender_20/_best_model'


MultVAERecommender: Saving complete
MultVAERecommender: Epoch 163 of 500. Elapsed time 1.22 hour


MultVAERecommender: Validation begins...


In [ ]:
if config['save_github'] and config['tune_parameters']:
    upload_file(
        config['database_path'], 
        f'tuning_results/recall@{config["candidates"]}/{config["database_name"]}',
        'Tuning db updated results (from kaggle notebook)'
    )

    if config['save_json']:
        j_name = config['json_name']
        l_path = f"/kaggle/working/{j_name}"
        r_path = f"tuning_results/recall@{config['candidates']}/best_params/{j_name}"
        
        best_val, best_params, best_trial_num, best_study_name = -float('inf'), {}, None, None
        
        try:
            for s in optuna.get_all_study_summaries(storage=f"sqlite:///{config['database_path']}"):
                if s.best_trial and s.best_trial.value > best_val:
                    best_val, best_params, best_trial_num, best_study_name = s.best_trial.value, s.best_trial.params, s.best_trial.number, s.study_name
        except:
            best_val, best_params, best_trial_num, best_study_name = study.best_value, study.best_params, study.best_trial.number, study.study_name
    
        prev_val, exists = -float('inf'), False
        try:
            prev_val = json.loads(repo.get_contents(r_path).decoded_content.decode()).get('value', -float('inf'))
            exists = True
        except:
            pass
    
        if not exists or best_val > prev_val:
            print(f"Updating JSON. Best: {best_val:.6f} (Prev: {prev_val:.6f})")
            best_params['epochs'] = study.best_trial.user_attrs['epochs']
            with open(l_path, 'w') as f:
                json.dump({"value": best_val, "study_name": best_study_name, "trial_number": best_trial_num, "params": best_params}, f, indent=4)
            if config.get('save_github'):
                upload_file(l_path, r_path, f'Update params {j_name} (from kaggle notebook)')
        else:
            print(f"No update needed. Best: {best_val:.6f} <= Saved: {prev_val:.6f}")